# Discovery, Validation, And Artifact Inspection

This notebook uses the shared Drive dataset as a read-only source, uses local Colab artifacts for reliability, reuses the training notebook runtime config when available, and exports the finished discovery/validation outputs to a clearly separate Drive folder.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import json
import pandas as pd
from datetime import datetime

drive.mount('/content/drive')
repo_root = Path('/content/predictive_circuit_coding')
github_repo_url = 'https://github.com/jacobposchl/predictive_circuit_coding'

# Shared folder from the storage-heavy Drive account: read dataset from here.
shared_drive_root = Path('/content/drive/MyDrive/pcc_runs')
drive_data_root = shared_drive_root / 'data' / 'allen_visual_behavior_neuropixels'

# Keep exported Colab outputs in a clearly different folder name so they do not get
# confused with the source dataset bundle.
drive_export_root = Path('/content/drive/MyDrive/pcc_colab_outputs')
drive_export_root.mkdir(parents=True, exist_ok=True)

print('Repo root:', repo_root)
print('GitHub repo:', github_repo_url)
print('Shared dataset root:', drive_data_root)
print('Drive export root:', drive_export_root)

In [ ]:
if not repo_root.exists():
    !git clone {github_repo_url} {repo_root}
%cd {repo_root}
# Keep Colab installs minimal to avoid restart prompts from preloaded widget packages.
!pip install -e .

In [ ]:
repo_dataset_root = repo_root / 'data' / 'allen_visual_behavior_neuropixels'
repo_artifacts_root = repo_root / 'artifacts'

assert drive_data_root.exists(), f'Missing Drive dataset bundle: {drive_data_root}'

repo_dataset_root.parent.mkdir(parents=True, exist_ok=True)
if repo_dataset_root.exists() or repo_dataset_root.is_symlink():
    if repo_dataset_root.is_symlink():
        repo_dataset_root.unlink()
    else:
        shutil.rmtree(repo_dataset_root)
repo_dataset_root.symlink_to(drive_data_root, target_is_directory=True)

if not repo_artifacts_root.exists():
    repo_artifacts_root.mkdir(parents=True, exist_ok=True)

print('Linked dataset into repo:', repo_dataset_root)
print('Using local artifact directory:', repo_artifacts_root)

In [ ]:
import subprocess
import yaml
from predictive_circuit_coding.utils import (
    NotebookDatasetConfig,
    NotebookStageReporter,
    output_indicates_missing_positive_labels,
    prepare_notebook_runtime_context,
    resolve_notebook_checkpoint,
    restore_latest_exported_artifacts,
    run_streaming_command,
    verify_paths_exist,
)

NOTEBOOK_STEP_LOG_EVERY = 16
DISCOVERY_SPLIT_CANDIDATES = ['discovery', 'valid', 'test']
NOTEBOOK_DATASET = NotebookDatasetConfig(
    use_full_dataset=False,
    experience_level='Familiar',
    max_sessions=10,
)

reporter = NotebookStageReporter(name='colab-discover', expected_duration='1-5 minutes for debug runs, longer for larger discovery passes')
reporter.banner('Discovery And Validation', subtitle='Frozen-token discovery, validation controls, inspection, and export')

base_experiment_config = repo_root / 'configs' / 'pcc' / 'predictive_circuit_coding_base.yaml'
data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
runtime_experiment_config = repo_root / 'colab_runtime_experiment.yaml'
checkpoint_dir = repo_root / 'artifacts' / 'checkpoints'
summary_path = repo_root / 'artifacts' / 'training_summary.json'
selection_output_name = 'unknown'
selected_session_count = None
restored_export_run = None

if not summary_path.exists() and not checkpoint_dir.exists():
    restored_export_run = restore_latest_exported_artifacts(
        drive_export_root=drive_export_root,
        local_artifact_root=repo_root / 'artifacts',
        runtime_experiment_config=runtime_experiment_config,
    )
elif not summary_path.exists() and checkpoint_dir.exists() and not any(checkpoint_dir.glob('*.pt')):
    restored_export_run = restore_latest_exported_artifacts(
        drive_export_root=drive_export_root,
        local_artifact_root=repo_root / 'artifacts',
        runtime_experiment_config=runtime_experiment_config,
    )

if runtime_experiment_config.exists():
    experiment_config = runtime_experiment_config
    runtime_payload = yaml.safe_load(experiment_config.read_text(encoding='utf-8'))
    dataset_selection_active = any(
        value not in (None, [], '', {})
        for key, value in runtime_payload.get('dataset_selection', {}).items()
        if key != 'output_name'
    )
    selection_output_name = runtime_payload.get('dataset_selection', {}).get('output_name', 'unknown')
else:
    runtime_context = prepare_notebook_runtime_context(
        base_experiment_config=base_experiment_config,
        runtime_experiment_config=runtime_experiment_config,
        session_catalog_csv=repo_root / 'data' / 'allen_visual_behavior_neuropixels' / 'manifests' / 'session_catalog.csv',
        artifact_root=repo_root / 'artifacts',
        dataset_config=NOTEBOOK_DATASET,
        step_log_every=NOTEBOOK_STEP_LOG_EVERY,
    )
    experiment_config = runtime_context.experiment_config_path
    checkpoint_dir = runtime_context.checkpoint_dir
    summary_path = runtime_context.summary_path
    dataset_selection_active = runtime_context.dataset_selection_active
    selection_output_name = runtime_context.selection_output_name
    selected_session_count = runtime_context.selected_session_count

selected_checkpoint = resolve_notebook_checkpoint(summary_path=summary_path, checkpoint_dir=checkpoint_dir)
run_label = datetime.now().strftime('discover_run_%Y%m%d_%H%M%S')
drive_run_export_root = drive_export_root / run_label

print('Experiment config:', experiment_config)
print('Data config:', data_config)
print('Runtime selection active:', dataset_selection_active)
print('Selection output name:', selection_output_name)
if selected_session_count is not None:
    print('Selected sessions:', selected_session_count)
if restored_export_run is not None:
    print('Restored exported training run:', restored_export_run)
print('Notebook step-log cadence:', NOTEBOOK_STEP_LOG_EVERY)
print('Discovery split candidates:', DISCOVERY_SPLIT_CANDIDATES)
print('Resolved checkpoint dir:', checkpoint_dir.resolve())
print('Resolved summary path:', summary_path.resolve())
print('Selected checkpoint:', selected_checkpoint)
print('Drive export path:', drive_run_export_root)

In [ ]:
paths_ok = verify_paths_exist({
    'experiment_config': experiment_config,
    'data_config': data_config,
    'checkpoint': selected_checkpoint,
    'drive_dataset_root': drive_data_root,
})
print(paths_ok)
assert all(paths_ok.values()), 'Missing discovery/validation inputs.'

if dataset_selection_active:
    reporter.begin('runtime selection', next_artifact='selected split manifest')
    run_streaming_command([
        'pcc-prepare-data',
        'materialize-runtime-selection',
        '--config', str(experiment_config),
        '--data-config', str(data_config),
    ], cwd=repo_root, step_log_every=NOTEBOOK_STEP_LOG_EVERY)
    reporter.finish('runtime selection')
else:
    print('Using the full canonical dataset. No runtime subset will be materialized.')

In [ ]:
reporter.begin('discovery', next_artifact='local discovery artifact + cluster summary')
%cd {repo_root}
resolved_discovery_split = None
last_discovery_error = None
for candidate_split in DISCOVERY_SPLIT_CANDIDATES:
    print(f'Trying discovery split: {candidate_split}')
    try:
        run_streaming_command([
            'pcc-discover',
            '--config', str(experiment_config),
            '--data-config', str(data_config),
            '--checkpoint', str(selected_checkpoint),
            '--split', candidate_split,
        ], cwd=repo_root, step_log_every=NOTEBOOK_STEP_LOG_EVERY)
        resolved_discovery_split = candidate_split
        break
    except subprocess.CalledProcessError as exc:
        last_discovery_error = exc
        output = exc.output or ''
        if output_indicates_missing_positive_labels(output):
            print(f'No positive target-label windows were found on split {candidate_split}; trying the next split.')
            continue
        raise
if resolved_discovery_split is None:
    if last_discovery_error is not None:
        raise last_discovery_error
    raise RuntimeError('Discovery did not complete and no split fallback succeeded.')
print('Resolved discovery split:', resolved_discovery_split)
reporter.finish('discovery')

In [ ]:
discovery_artifact = Path(selected_checkpoint).with_name(f"{Path(selected_checkpoint).stem}_{resolved_discovery_split}_discovery.json")
cluster_summary_json = discovery_artifact.with_name(f"{discovery_artifact.stem}_cluster_summary.json")
cluster_summary_csv = discovery_artifact.with_name(f"{discovery_artifact.stem}_cluster_summary.csv")

reporter.begin('validation', next_artifact='local validation summary json/csv')
%cd {repo_root}
run_streaming_command([
    'pcc-validate',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
    '--checkpoint', str(selected_checkpoint),
    '--discovery-artifact', str(discovery_artifact),
], cwd=repo_root, step_log_every=NOTEBOOK_STEP_LOG_EVERY)
reporter.finish('validation')

In [ ]:
validation_json = discovery_artifact.with_name(f"{discovery_artifact.stem}_validation.json")
validation_csv = discovery_artifact.with_name(f"{discovery_artifact.stem}_validation.csv")

with open(discovery_artifact, 'r', encoding='utf-8') as handle:
    discovery_payload = json.load(handle)
with open(cluster_summary_json, 'r', encoding='utf-8') as handle:
    cluster_summary_payload = json.load(handle)
with open(validation_json, 'r', encoding='utf-8') as handle:
    validation_payload = json.load(handle)

display(pd.read_csv(cluster_summary_csv))
display(pd.DataFrame(discovery_payload['candidates']).head())
display(pd.read_csv(validation_csv))
print('Cluster count:', cluster_summary_payload['cluster_count'])
print('Recurrence summary:', validation_payload['recurrence_summary'])

In [ ]:
reporter.begin('export', next_artifact='Drive copy of local artifacts')
if drive_run_export_root.exists():
    shutil.rmtree(drive_run_export_root)
shutil.copytree(repo_artifacts_root, drive_run_export_root)
print('Exported local artifacts to:', drive_run_export_root)
reporter.finish('export')